# Kaggle Quick Try: Phi-3 Mini, Llama 8B, Mixtral 8x22B

This notebook is Kaggle-friendly and intentionally minimal.

Before running model cells:
- Enable GPU in Kaggle (Notebook settings).
- Add a Kaggle secret named `HF_TOKEN` with your Hugging Face token.
- Ensure your token has access to gated models (Llama).

Note: Mixtral 8x22B is called via Hugging Face hosted inference because full local loading is too large for standard Kaggle GPU memory.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

In [ ]:
import gc
import os

import torch
from huggingface_hub import InferenceClient
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

HF_TOKEN = os.getenv("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN in Kaggle Secrets (or as env var) before running model cells.")

prompt = (
    "You are a routing assistant. Given user intent: 'Find the fastest train from Paris to Berlin tomorrow morning', "
    "return concise reasoning and the best action plan."
)

print("Setup complete.")

In [ ]:
# Phi-3 Mini (3.8B)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

phi_model_id = "microsoft/Phi-3-mini-4k-instruct"
phi_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

phi_tokenizer = AutoTokenizer.from_pretrained(
    phi_model_id,
    token=HF_TOKEN,
    trust_remote_code=True
)
phi_model = AutoModelForCausalLM.from_pretrained(
    phi_model_id,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=phi_bnb,
    trust_remote_code=True
)

phi_generator = pipeline("text-generation", model=phi_model, tokenizer=phi_tokenizer)
phi_result = phi_generator(
    prompt,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.2,
    top_p=0.9
)
print(phi_result[0]["generated_text"])

In [ ]:
# Llama 8B (practical Kaggle choice: Llama-3.1-8B-Instruct)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

llama_model_id = "meta-llama/Llama-3.1-8B-Instruct"
llama_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_id, token=HF_TOKEN)
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_id,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=llama_bnb
)

llama_generator = pipeline("text-generation", model=llama_model, tokenizer=llama_tokenizer)
llama_result = llama_generator(
    prompt,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.2,
    top_p=0.9
)
print(llama_result[0]["generated_text"])

In [ ]:
# Mixtral 8x22B via hosted inference (local load is generally too large for Kaggle GPUs)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

mixtral_model_id = "mistralai/Mixtral-8x22B-Instruct-v0.1"
mixtral_client = InferenceClient(token=HF_TOKEN)
mixtral_prompt = f"<s>[INST] {prompt} [/INST]"

mixtral_result = mixtral_client.text_generation(
    prompt=mixtral_prompt,
    model=mixtral_model_id,
    max_new_tokens=220,
    temperature=0.2,
    top_p=0.9
)
print(mixtral_result)